In [1]:
%pip install -q "transformers>=4.41.0" datasets accelerate scikit-learn pandas numpy sentencepiece protobuf

In [2]:
import inspect
import json
import math
import os
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Mount Google Drive va cau hinh duong dan

Neu project cua ban khong nam o mot trong cac duong dan mac dinh, hay sua `PROJECT_DIR_CANDIDATES`.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")

DATA_RELATIVE_PATH = Path("data/processed/student_voice_enriched_reviewed.csv")

PROJECT_DIR_CANDIDATES = [
    DRIVE_ROOT / "Student Voice Intelligence",
    DRIVE_ROOT / "AI thuc chien" / "Student Voice Intelligence",
    DRIVE_ROOT / "Colab Notebooks" / "Student Voice Intelligence",
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / DATA_RELATIVE_PATH).exists():
        PROJECT_DIR = candidate
        break


DATA_PATH = PROJECT_DIR / DATA_RELATIVE_PATH

MODEL_ROOT = PROJECT_DIR / "outputs/models/transformer"
REPORT_DIR = PROJECT_DIR / "outputs/reports/transformer/videberta"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = f"videberta_base_sentiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = MODEL_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MODEL_DIR = RUN_DIR / "model"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PATH:", DATA_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("REPORT_DIR:", REPORT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Student Voice Intelligence
DATA_PATH: /content/drive/MyDrive/Student Voice Intelligence/data/processed/student_voice_enriched_reviewed.csv
MODEL_DIR: /content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/videberta_base_sentiment_20260620_080117/model
REPORT_DIR: /content/drive/MyDrive/Student Voice Intelligence/outputs/reports/transformer/videberta


## 2. Cau hinh train

Cau hinh nay uu tien do on dinh cho ViDeBERTa tren Colab T4. ViDeBERTa da tung bi `nan loss` voi fp16, nen notebook nay dung float32 va learning rate thap hon.

In [5]:
MODEL_NAME = "Fsoft-AIC/videberta-base"
TEXT_COL = "text"
TARGET_COL = "sentiment_std_3class"
SPLIT_COL = "split_original"

LABELS = ["negative", "neutral", "positive"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

MAX_LENGTH = 192
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01

BASELINE_TEST_MACRO_F1 = 0.811936
XLMR_TEST_MACRO_F1 = 0.852064
PHOBERT_V2_TEST_MACRO_F1 = 0.857618

print("label2id:", label2id)

label2id: {'negative': 0, 'neutral': 1, 'positive': 2}


## 3. Load data va lam sach nhe


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

required_cols = [TEXT_COL, TARGET_COL, SPLIT_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

work_df = df[required_cols].copy()
work_df[TEXT_COL] = work_df[TEXT_COL].fillna("").astype(str).str.strip()
work_df[TARGET_COL] = work_df[TARGET_COL].fillna("").astype(str).str.strip()
work_df[SPLIT_COL] = work_df[SPLIT_COL].fillna("").astype(str).str.strip()

work_df = work_df[
    (work_df[TEXT_COL] != "")
    & (work_df[TARGET_COL].isin(LABELS))
    & (work_df[SPLIT_COL].isin(["train", "validation", "test"]))
].copy()

train_df = work_df.loc[work_df[SPLIT_COL] == "train"].copy()
conflict_counts = train_df.groupby(TEXT_COL)[TARGET_COL].nunique()
conflict_texts = set(conflict_counts[conflict_counts > 1].index)

before_rows = len(work_df)
work_df = work_df[~((work_df[SPLIT_COL] == "train") & (work_df[TEXT_COL].isin(conflict_texts)))].copy()

work_df["labels"] = work_df[TARGET_COL].map(label2id).astype(int)

print("Shape:", work_df.shape)
print("Split distribution:")
display(pd.crosstab(work_df[SPLIT_COL], work_df[TARGET_COL], margins=True))

Shape: (49139, 4)
Split distribution:


sentiment_std_3class,negative,neutral,positive,All
split_original,,,,
test,2630,4725,2424,9779
train,9539,16394,8539,34472
validation,1314,2352,1222,4888
All,13483,23471,12185,49139


## 4. Tao Hugging Face Dataset va tokenize

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf_dataset(split_name: str) -> Dataset:
    split_df = work_df.loc[work_df[SPLIT_COL] == split_name, [TEXT_COL, "labels"]].reset_index(drop=True)
    if split_df.empty:
        raise ValueError(f"Split rong: {split_name}")
    return Dataset.from_pandas(split_df, preserve_index=False)


raw_datasets = DatasetDict(
    {
        "train": to_hf_dataset("train"),
        "validation": to_hf_dataset("validation"),
        "test": to_hf_dataset("test"),
    }
)


def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_datasets = raw_datasets.map(tokenize_batch, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns([TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.49M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/34472 [00:00<?, ? examples/s]

Map:   0%|          | 0/4888 [00:00<?, ? examples/s]

Map:   0%|          | 0/9779 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 34472
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4888
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9779
    })
})


## 5. Khoi tao model va metric

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32,
)
model = model.float()


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/567M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: Fsoft-AIC/videberta-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those pa

## 6. Train

Cell nay tu kiem tra version `transformers` de tranh loi tham so `tokenizer` cua `Trainer`.

In [9]:
steps_per_epoch = math.ceil(len(tokenized_datasets["train"]) / TRAIN_BATCH_SIZE)
warmup_steps = int(steps_per_epoch * NUM_EPOCHS * 0.06)

training_args_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": warmup_steps,
    "max_grad_norm": 1.0,
    "logging_steps": 50,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "fp16": False,
    "report_to": "none",
    "seed": SEED,
}

training_args_signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_args_signature.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

if "bf16" in training_args_signature.parameters:
    training_args_kwargs["bf16"] = False

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__)
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

train_result = trainer.train()
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))


model.safetensors:   0%|          | 0.00/567M [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.747564,0.784371,0.681669,0.615733,0.648739
2,0.660713,0.691532,0.731588,0.699892,0.721762
3,0.637016,0.677918,0.735065,0.711354,0.729622


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/videberta_base_sentiment_20260620_080117/model/tokenizer_config.json',
 '/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/videberta_base_sentiment_20260620_080117/model/tokenizer.json')

## 7. Evaluate validation/test va luu report

In [10]:
def evaluate_split(split_name: str) -> dict:
    predictions = trainer.predict(tokenized_datasets[split_name])
    y_true = predictions.label_ids
    y_pred = np.argmax(predictions.predictions, axis=-1)

    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            target_names=LABELS,
            output_dict=True,
            zero_division=0,
        )
    ).T
    cm_df = pd.DataFrame(confusion_matrix(y_true, y_pred), index=LABELS, columns=LABELS)

    report_df.to_csv(REPORT_DIR / f"{split_name}_classification_report.csv", encoding="utf-8-sig")
    cm_df.to_csv(REPORT_DIR / f"{split_name}_confusion_matrix.csv", encoding="utf-8-sig")

    return metrics


validation_metrics = evaluate_split("validation")
test_metrics = evaluate_split("test")

all_metrics = {
    "model_name": MODEL_NAME,
    "target_col": TARGET_COL,
    "text_col_used": TEXT_COL,
    "labels": LABELS,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": warmup_steps,
    "baseline_test_macro_f1": BASELINE_TEST_MACRO_F1,
    "xlmr_test_macro_f1": XLMR_TEST_MACRO_F1,
    "phobert_v2_test_macro_f1": PHOBERT_V2_TEST_MACRO_F1,
    "validation": validation_metrics,
    "test": test_metrics,
    "model_dir": str(MODEL_DIR),
    "report_dir": str(REPORT_DIR),
}

with open(REPORT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)

with open(MODEL_ROOT / "videberta_base_sentiment_latest.txt", "w", encoding="utf-8") as f:
    f.write(str(MODEL_DIR))

print(json.dumps(all_metrics, ensure_ascii=False, indent=2))
print("Reports:", REPORT_DIR)

{
  "model_name": "Fsoft-AIC/videberta-base",
  "target_col": "sentiment_std_3class",
  "text_col_used": "text",
  "labels": [
    "negative",
    "neutral",
    "positive"
  ],
  "max_length": 192,
  "train_batch_size": 8,
  "eval_batch_size": 16,
  "num_epochs": 3,
  "learning_rate": 1e-05,
  "warmup_steps": 775,
  "baseline_test_macro_f1": 0.811936,
  "xlmr_test_macro_f1": 0.852064,
  "phobert_v2_test_macro_f1": 0.857618,
  "validation": {
    "split": "validation",
    "accuracy": 0.7350654664484452,
    "macro_f1": 0.7113541409005079,
    "weighted_f1": 0.7296219007913022
  },
  "test": {
    "split": "test",
    "accuracy": 0.7351467430207588,
    "macro_f1": 0.7097950336594421,
    "weighted_f1": 0.7293181772170259
  },
  "model_dir": "/content/drive/MyDrive/Student Voice Intelligence/outputs/models/transformer/videberta_base_sentiment_20260620_080117/model",
  "report_dir": "/content/drive/MyDrive/Student Voice Intelligence/outputs/reports/transformer/videberta"
}
Reports: /con

## 8. Test nhanh vai cau thuc te

In [11]:
infer_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
infer_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)
infer_model.eval()


def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]

    inputs = infer_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = infer_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

    rows = []
    for text, prob in zip(texts, probs):
        pred_id = int(np.argmax(prob))
        row = {
            "text": text,
            "prediction": id2label[pred_id],
            "confidence": float(prob[pred_id]),
        }
        for idx, label in id2label.items():
            row[f"prob_{label}"] = float(prob[idx])
        rows.append(row)

    return pd.DataFrame(rows)


sample_texts = [
    "Thầy giảng rất dễ hiểu, nhiệt tình hỗ trợ sinh viên.",
    "Wifi phòng học quá yếu, máy chiếu bị mờ nên rất khó chịu.",
    "Em muốn hỏi lịch thi cuối kỳ môn này khi nào có ạ?",
    "Cán bộ phòng đào tạo phản hồi quá chậm, em không biết hỏi ai.",
    "Nội dung môn học khá ổn nhưng bài tập hơi nhiều.",
    "Giảng viên đến muộn nhiều buổi và giải thích rất khó hiểu.",
]

predict_sentiment(sample_texts)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

,text,prediction,confidence,prob_negative,prob_neutral,prob_positive
0,"Thầy giảng rất dễ hiểu, nhiệt tình hỗ trợ sinh...",positive,0.994108,0.004397,0.001496,0.994108
1,"Wifi phòng học quá yếu, máy chiếu bị mờ nên rấ...",negative,0.957254,0.957254,0.016708,0.026038
2,Em muốn hỏi lịch thi cuối kỳ môn này khi nào c...,neutral,0.631730,0.328538,0.631730,0.039732
3,"Cán bộ phòng đào tạo phản hồi quá chậm, em khô...",neutral,0.541127,0.288859,0.541127,0.170014
4,Nội dung môn học khá ổn nhưng bài tập hơi nhiều.,negative,0.938274,0.938274,0.019844,0.041882
5,Giảng viên đến muộn nhiều buổi và giải thích r...,negative,0.934279,0.934279,0.013882,0.051839
